<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 8 · Miércoles — XGBoost</h1>
<h3>El campeón de las competencias Kaggle</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## Objetivos

Al final de la clase van a poder:

1. Entender qué hace distinto a XGBoost de LightGBM (a pesar de que ambos son boosting).
2. Dominar los hiperparámetros que importan en XGBoost.
3. Usar XGBoost para clasificación (no solo regresión como ayer).
4. Combinar XGBoost con Optuna para tuning profesional.
5. Resolver UN ejercicio largo donde XGBoost compita contra LightGBM y Random Forest sobre un problema real.

# 1. Historia rápida: XGBoost antes que LightGBM

XGBoost salió en 2014, 3 años antes que LightGBM. Fue una revolución: durante esos años, ganó prácticamente todas las competencias de Kaggle de datos tabulares. Hoy día sigue siendo uno de los modelos más usados en industria.

Cuando salió LightGBM en 2017, XGBoost dejó de ser el rey por velocidad, pero sigue siendo competitivo en precisión. La diferencia hoy es marginal:

| | XGBoost | LightGBM |
|---|---|---|
| Año | 2014 | 2017 |
| Crecimiento del árbol | Level-wise (por nivel) | Leaf-wise (por hoja) |
| Velocidad | Más lento | Más rápido (~10x) |
| Manejo de categóricas nativas | Soportado desde versión 1.3 | Sí (categorical_feature) |
| Performance | Excelente | Excelente |
| Madurez | Muy maduro | Maduro |

Mi recomendación práctica: si tienes que elegir uno, **LightGBM** por velocidad. Pero XGBoost sigue vigente y vale la pena conocerlo porque mucho código legacy lo usa, y a veces saca un pelín mejor performance.

Instalación: `pip install xgboost`

# 2. Diferencia clave: level-wise vs leaf-wise

Esta es la diferencia más importante entre XGBoost y LightGBM:

**Level-wise (XGBoost)**: en cada paso, crece todas las hojas del mismo nivel. Resultado: árboles más balanceados, más predecibles, menos riesgo de overfitting.

**Leaf-wise (LightGBM)**: en cada paso, elige la hoja que va a reducir más el error y la crece. Resultado: árboles más profundos en algunas ramas, más eficiente, pero mayor riesgo de overfitting con datasets chicos.

En términos prácticos:
- XGBoost gana en datasets chicos (menos de 10k filas) por su menor tendencia a overfittear.
- LightGBM gana en datasets grandes (más de 100k filas) por velocidad.

# 3. Hiperparámetros clave de XGBoost

Casi los mismos que LightGBM pero con nombres ligeramente distintos:

| XGBoost | LightGBM | Qué hace | Rango típico |
|---|---|---|---|
| `n_estimators` | `n_estimators` | Número de árboles | 100-1000 |
| `learning_rate` (o `eta`) | `learning_rate` | Tamaño de paso | 0.01-0.3 |
| `max_depth` | `max_depth` | Profundidad del árbol | 3-10 |
| `min_child_weight` | `min_child_samples` | Min muestras por hoja | 1-10 |
| `subsample` | `subsample` | % filas por árbol | 0.6-1.0 |
| `colsample_bytree` | `colsample_bytree` | % features por árbol | 0.6-1.0 |
| `gamma` | `min_gain_to_split` | Min ganancia para dividir | 0-5 |
| `reg_alpha` | `reg_alpha` | Regularización L1 | 0-10 |
| `reg_lambda` | `reg_lambda` | Regularización L2 | 0-10 |

El más importante: `learning_rate` y `n_estimators` van juntos. Si subes uno, baja el otro.

## Setup — esta vez con clasificación

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report)

import xgboost as xgb
import lightgbm as lgb

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

# Dataset: Breast Cancer (clasificación binaria, 569 pacientes, 30 features)
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Distribución de clases en train: {y_train.value_counts().to_dict()}')
print(f'(0 = maligno, 1 = benigno)')

# 4. XGBoost básico

In [ ]:
modelo_xgb = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

t0 = time.time()
modelo_xgb.fit(X_train, y_train)
t_xgb = time.time() - t0

pred = modelo_xgb.predict(X_test)
proba = modelo_xgb.predict_proba(X_test)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, pred):.4f}')
print(f'F1:       {f1_score(y_test, pred):.4f}')
print(f'AUC:      {roc_auc_score(y_test, proba):.4f}')
print(f'Tiempo:   {t_xgb:.2f}s')

# 5. Early stopping con XGBoost

Igual que LightGBM, XGBoost soporta early stopping.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

modelo_es = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    early_stopping_rounds=50,
    random_state=42,
    eval_metric='logloss'
)
modelo_es.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f'Árboles que realmente entrenó (best_iteration): {modelo_es.best_iteration}')
print(f'Accuracy test: {modelo_es.score(X_test, y_test):.4f}')

# 6. Feature Importance

XGBoost ofrece tres tipos de importance:

- `weight`: cuántas veces se usó cada feature (igual al `split` de LightGBM).
- `gain`: ganancia promedio por feature (la más usada para interpretar).
- `cover`: cuántas muestras cubre cada feature en promedio.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
xgb.plot_importance(modelo_xgb, max_num_features=10, importance_type='gain',
                     ax=ax, title='Top 10 Features por GAIN — Breast Cancer')
plt.tight_layout(); plt.show()

# 7. Tuning de XGBoost con Optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0, 10),
    }
    modelo = xgb.XGBClassifier(**params, random_state=42, eval_metric='logloss', verbosity=0)
    return cross_val_score(modelo, X_train, y_train, cv=3, scoring='f1', n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=False)

print(f'Mejor F1 CV: {study.best_value:.4f}')
print(f'Mejores params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

modelo_tuneado = xgb.XGBClassifier(**study.best_params, random_state=42,
                                     eval_metric='logloss', verbosity=0).fit(X_train, y_train)
print(f'\nMétricas en test:')
print(f'  Accuracy: {modelo_tuneado.score(X_test, y_test):.4f}')
print(f'  F1:       {f1_score(y_test, modelo_tuneado.predict(X_test)):.4f}')

---
# Ejercicio integrador

Un solo ejercicio, exigente. La idea es comparar XGBoost contra los modelos que ya conocen.

## Reto: detección de cáncer de mama con 4 modelos

Dataset: `load_breast_cancer` (569 pacientes, 30 features, target binario maligno/benigno).

Compitiendo: Logistic Regression, Random Forest, LightGBM, XGBoost tuneado.

### Lo que tienen que hacer

**Parte A — 3 baselines (15 min)**
1. Entrenar 3 modelos con parámetros por defecto: LogisticRegression (con StandardScaler), RandomForestClassifier y LGBMClassifier.
2. Para cada uno reportar: Accuracy, F1, Recall (clase maligno), AUC y tiempo de entrenamiento.
3. Importante: la clase 0 es maligno (cáncer). En este problema el RECALL de la clase 0 es lo que más importa: no se nos puede escapar ningún paciente con cáncer.

**Parte B — XGBoost tuneado con Optuna (25 min)**
4. Definir un objective con los 9 hiperparámetros principales de XGBoost.
5. Optimizar F1 con cv=5 y n_trials=30.
6. Reportar mejores hiperparámetros y métricas en test.
7. Graficar `optuna.visualization.plot_param_importances(study)` para ver cuáles hiperparámetros importan más.
8. Graficar `optuna.visualization.plot_optimization_history(study)` para ver la mejora trial a trial.

**Parte C — Matriz de confusión + análisis crítico (15 min)**
9. Mostrar la matriz de confusión del XGBoost tuneado.
10. Contar TP, TN, FP, FN. **¿Cuántos pacientes con cáncer (clase 0) se le escaparon al modelo (FN)?** Eso es lo que importa en medicina.
11. Si el modelo tiene FN > 0, ¿es aceptable para uso clínico? ¿Qué harían para mejorarlo?

**Parte D — Feature Importance + interpretación médica (15 min)**
12. Graficar las top 10 features según `xgb.plot_importance` con `importance_type='gain'`.
13. Las features están relacionadas con mediciones físicas del tumor (radio, textura, perímetro, área, etc).
14. ¿Qué tipo de mediciones dominan? ¿Tiene sentido médicamente?

**Parte E — Decisión final**
15. Tabla comparativa de los 4 modelos (Logistic, RF, LightGBM, XGBoost tuneado) con todas las métricas.
16. Si tuvieran que entregar UN modelo a un hospital, ¿cuál sería? Justifiquen pensando en:
   - Recall (no perder casos de cáncer).
   - Explicabilidad (los médicos quieren entender).
   - Velocidad de predicción (en consulta real).

### Bonus que vale puntos extra

- Comparar XGBoost con LightGBM tuneado (ambos con Optuna). ¿Quién gana?
- Probar `early_stopping_rounds` en XGBoost y comparar con el modelo sin early stopping.
- Ajustar el umbral de decisión del modelo ganador para maximizar Recall (sacrificando Precision si es necesario). ¿Cuánto sube el Recall? ¿Cuántas alarmas falsas extra genera?

In [ ]:
# Parte A — 3 baselines



In [ ]:
# Parte B — XGBoost tuneado con Optuna



In [ ]:
# Parte C — Matriz de confusión + análisis crítico



In [ ]:
# Parte D — Feature Importance + interpretación médica



In [ ]:
# Parte E — Decisión final



## Cierre

Hoy aprendimos:

- XGBoost: el campeón histórico de Kaggle, más maduro que LightGBM.
- La diferencia entre level-wise (XGBoost) y leaf-wise (LightGBM) y cuándo conviene cada uno.
- XGBoost para clasificación con ejemplos médicos reales.
- Early stopping y feature importance también funcionan acá.
- Combinación XGBoost + Optuna para tuning profesional.

Esta semana cerramos las técnicas de modelos. Mañana y viernes les toca a ustedes: van a defender un proyecto frente a un panel tipo VC.